## Project1: Airport Taxi Pickup Demand and Flight Arrivals Schedules

### Research Question

How are flight arrival schedules associated with taxi pickup demand at major NYC airports, and how can this relationship support practical driver positioning recommendations?

# 01 — NYC Taxi Data Preparation

### Purpose of this Notebook

This notebook prepares and explores the NYC TLC taxi trip data used in the project. The main objectives are to:

1. load the selected NYC TLC trip records using PySpark;
2. inspect the dataset structure and relevant attributes;
3. identify trips associated with major NYC airports;
4. clean and validate the relevant taxi records;
5. construct airport taxi demand measures for later integration with flight schedule data.

The detailed preprocessing steps and changes in dataset shape will be recorded to ensure reproducibility.

In [1]:
# Core PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Python utilities
from pathlib import Path
import pandas as pd
import numpy as np

# Start or retrieve the Spark session
spark = (
    SparkSession.builder
    .appName("MAST30034_Project1_AirportTaxiDemand")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 14:56:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


## 1. Study Design

This study focuses on Yellow Taxi trips associated with John F. Kennedy International Airport (JFK) and LaGuardia Airport (LGA).

The study period is January--June 2025, providing six months of taxi activity while keeping the large-scale PySpark workflow computationally manageable.

Taxi demand is initially defined as the number of airport taxi pickups per hour. This hourly demand measure will later be compared with airport flight activity, including potential time-lagged relationships between flight arrivals and taxi pickups.

In [2]:
# Project configuration

STUDY_START = "2025-01-01"
STUDY_END = "2025-06-30"

TAXI_TYPE = "yellow"

AIRPORTS = ["JFK", "LGA"]

print("Study period:", STUDY_START, "to", STUDY_END)
print("Taxi type:", TAXI_TYPE)
print("Airports:", AIRPORTS)

Study period: 2025-01-01 to 2025-06-30
Taxi type: yellow
Airports: ['JFK', 'LGA']


## 2. Load Yellow Taxi Trip Data

Loading the six monthly Yellow Taxi Trip Record datasets covering January to June 2025 using PySpark.

In [3]:
# Path to the six monthly Yellow Taxi parquet files
RAW_TAXI_PATH = "../data/raw/yellow_tripdata_2025-*.parquet"

# Load all six months into one Spark DataFrame
taxi_raw = spark.read.parquet(RAW_TAXI_PATH)

print("Yellow Taxi data loaded successfully.")
print("Number of columns:", len(taxi_raw.columns))

26/08/31 15:08:55 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/yellow_tripdata_2025-*.parquet.
java.io.FileNotFoundException: File ../data/raw/yellow_tripdata_2025-*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at 

Yellow Taxi data loaded successfully.
Number of columns: 20


In [4]:
# Inspect the dataset schema
taxi_raw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
# Inspect a small sample of records
taxi_raw.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type"
).show(10, truncate=False)

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+----------+------------+------------+
|2025-01-01 00:18:38 |2025-01-01 00:26:59  |229         |237         |1              |1.6          |10.0       |3.0       |18.0        |1           |
|2025-01-01 00:32:40 |2025-01-01 00:35:13  |236         |237         |1              |0.5          |5.1        |2.02      |12.12       |1           |
|2025-01-01 00:44:04 |2025-01-01 00:46:01  |141         |141         |1              |0.6          |5.1        |2.0       |12.1        |1           |
|2025-01-01 00:14:27 |2025-01-01 00:20:01  |244         |244         |3              |0.52         |

## 3. Raw Dataset Profiling

Before applying any filtering, we inspect the full six-month Yellow Taxi dataset to establish the baseline dataset shape and identify potential data-quality issues. This provides a reference point for documenting all subsequent preprocessing steps.

In [6]:
# Count the full raw dataset
raw_count = taxi_raw.count()

print("Raw dataset rows:", raw_count)
print("Raw dataset columns:", len(taxi_raw.columns))

Raw dataset rows: 24083384
Raw dataset columns: 20


In [7]:
# Check the observed pickup datetime range
taxi_raw.select(
    F.min("tpep_pickup_datetime").alias("min_pickup_datetime"),
    F.max("tpep_pickup_datetime").alias("max_pickup_datetime")
).show(truncate=False)

+-------------------+-------------------+
|min_pickup_datetime|max_pickup_datetime|
+-------------------+-------------------+
|2007-12-05 18:45:00|2025-06-30 23:59:59|
+-------------------+-------------------+



In [8]:
# Check missing values in variables relevant to this study
key_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "payment_type"
]

missing_summary = taxi_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in key_cols
])

missing_summary.show(truncate=False)

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|0                   |0                    |0           |0           |5418601        |0            |0          |0           |
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+



In [9]:
# Inspect basic ranges of key numerical variables
taxi_raw.select(
    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.min("fare_amount").alias("min_fare_amount"),
    F.max("fare_amount").alias("max_fare_amount"),
    F.min("passenger_count").alias("min_passenger_count"),
    F.max("passenger_count").alias("max_passenger_count")
).show(truncate=False)

+-----------------+-----------------+---------------+---------------+-------------------+-------------------+
|min_trip_distance|max_trip_distance|min_fare_amount|max_fare_amount|min_passenger_count|max_passenger_count|
+-----------------+-----------------+---------------+---------------+-------------------+-------------------+
|0.0              |386088.43        |-1807.6        |863372.12      |0                  |9                  |
+-----------------+-----------------+---------------+---------------+-------------------+-------------------+



In [10]:
# Count records outside the defined study period
out_of_period = taxi_raw.filter(
    (F.col("tpep_pickup_datetime") < F.lit(STUDY_START)) |
    (F.col("tpep_pickup_datetime") >= F.lit("2025-07-01"))
)

out_of_period_count = out_of_period.count()

print("Records outside study period:", out_of_period_count)
print(
    "Percentage outside study period:",
    round(out_of_period_count / raw_count * 100, 4),
    "%"
)

Records outside study period: 24
Percentage outside study period: 0.0001 %


In [11]:
# Inspect the prevalence of implausible numerical values
taxi_raw.select(
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("distance_le_0"),
    F.sum((F.col("trip_distance") > 100).cast("int")).alias("distance_gt_100"),
    F.sum((F.col("fare_amount") < 0).cast("int")).alias("fare_lt_0"),
    F.sum((F.col("fare_amount") > 500).cast("int")).alias("fare_gt_500"),
    F.sum((F.col("passenger_count") <= 0).cast("int")).alias("passengers_le_0"),
    F.sum((F.col("passenger_count") > 6).cast("int")).alias("passengers_gt_6")
).show(truncate=False)

+-------------+---------------+---------+-----------+---------------+---------------+
|distance_le_0|distance_gt_100|fare_lt_0|fare_gt_500|passengers_le_0|passengers_gt_6|
+-------------+---------------+---------+-----------+---------------+---------------+
|661842       |1488           |1323048  |381        |138981         |84             |
+-------------+---------------+---------+-----------+---------------+---------------+



In [12]:
# Inspect airport-related variables
taxi_raw.select(
    "PULocationID",
    "DOLocationID",
    "RatecodeID",
    "airport_fee"
).show(20, truncate=False)

+------------+------------+----------+-----------+
|PULocationID|DOLocationID|RatecodeID|airport_fee|
+------------+------------+----------+-----------+
|229         |237         |1         |0.0        |
|236         |237         |1         |0.0        |
|141         |141         |1         |0.0        |
|244         |244         |1         |0.0        |
|244         |116         |1         |0.0        |
|239         |68          |1         |0.0        |
|170         |170         |1         |0.0        |
|234         |148         |1         |0.0        |
|148         |170         |1         |0.0        |
|237         |262         |1         |0.0        |
|237         |75          |1         |0.0        |
|263         |236         |1         |0.0        |
|236         |151         |1         |0.0        |
|229         |141         |1         |0.0        |
|141         |113         |1         |0.0        |
|158         |170         |1         |0.0        |
|164         |229         |1   

In [13]:
# Distribution of airport-related indicators
taxi_raw.groupBy("RatecodeID") \
    .count() \
    .orderBy("RatecodeID") \
    .show(truncate=False)

taxi_raw.groupBy("airport_fee") \
    .count() \
    .orderBy("airport_fee") \
    .show(truncate=False)

+----------+--------+
|RatecodeID|count   |
+----------+--------+
|NULL      |5418601 |
|1         |17440674|
|2         |650668  |
|3         |64341   |
|4         |51425   |
|5         |193836  |
|6         |26      |
|99        |263813  |
+----------+--------+

+-----------+--------+
|airport_fee|count   |
+-----------+--------+
|NULL       |5418601 |
|-1.75      |78280   |
|0.0        |17049499|
|0.75       |1       |
|1.25       |12      |
|1.75       |1529795 |
|5.0        |1415    |
|6.75       |5781    |
+-----------+--------+



## 4. Airport Taxi Demand Construction

Airport taxi demand is defined using pickup location rather than fare- or rate-based airport indicators. A trip contributes to airport demand if its pickup location corresponds to the official TLC taxi zone for JFK or LaGuardia Airport.

This definition directly measures the number of Yellow Taxi departures from each airport and avoids relying on fare-related fields that contain substantial missingness or may reflect pricing rules rather than physical pickup location.

In [14]:
# TLC taxi-zone IDs for the airports included in the study
AIRPORT_ZONE_IDS = {
    132: "JFK",
    138: "LGA"
}

print("Airport taxi-zone mapping:")
for zone_id, airport in AIRPORT_ZONE_IDS.items():
    print(f"{airport}: PULocationID {zone_id}")

Airport taxi-zone mapping:
JFK: PULocationID 132
LGA: PULocationID 138


In [15]:
# Count raw Yellow Taxi pickups at JFK and LaGuardia
airport_pickup_counts = (
    taxi_raw
    .filter(F.col("PULocationID").isin(list(AIRPORT_ZONE_IDS.keys())))
    .groupBy("PULocationID")
    .count()
    .orderBy("PULocationID")
)

airport_pickup_counts.show(truncate=False)

+------------+------+
|PULocationID|count |
+------------+------+
|132         |975772|
|138         |634755|
+------------+------+



## 5. Airport Taxi Data Cleaning

The raw Yellow Taxi records are filtered to the January--June 2025 study period and to pickups originating from the JFK and LaGuardia TLC taxi zones.

Because the outcome of interest is the number of airport taxi pickups rather than trip revenue, records are not removed solely because of missing or unusual fare, passenger-count, rate-code, or airport-fee values. These fields are not required to establish that an airport pickup occurred.

Cleaning therefore focuses on variables required to construct a reliable airport-hour demand measure: pickup time and pickup location.

In [16]:
# Restrict records to the study period and airport pickup zones
airport_taxi = (
    taxi_raw
    .filter(
        (F.col("tpep_pickup_datetime") >= F.lit(STUDY_START)) &
        (F.col("tpep_pickup_datetime") < F.lit("2025-07-01"))
    )
    .filter(
        F.col("PULocationID").isin(list(AIRPORT_ZONE_IDS.keys()))
    )
)

airport_taxi_count = airport_taxi.count()

print("Airport taxi records after study-period and airport filtering:",
      airport_taxi_count)
print(
    "Records removed from raw dataset:",
    raw_count - airport_taxi_count
)

Airport taxi records after study-period and airport filtering: 1610523
Records removed from raw dataset: 22472861


In [17]:
# Check variables required for constructing hourly airport taxi demand
airport_taxi.select(
    F.sum(F.col("tpep_pickup_datetime").isNull().cast("int"))
        .alias("missing_pickup_datetime"),
    F.sum(F.col("PULocationID").isNull().cast("int"))
        .alias("missing_pickup_location")
).show(truncate=False)

+-----------------------+-----------------------+
|missing_pickup_datetime|missing_pickup_location|
+-----------------------+-----------------------+
|0                      |0                      |
+-----------------------+-----------------------+



In [18]:
# Add a human-readable airport label
airport_taxi = (
    airport_taxi
    .withColumn(
        "airport",
        F.when(F.col("PULocationID") == 132, F.lit("JFK"))
         .when(F.col("PULocationID") == 138, F.lit("LGA"))
    )
)

airport_taxi.groupBy("airport").count().orderBy("airport").show()

+-------+------+
|airport| count|
+-------+------+
|    JFK|975770|
|    LGA|634753|
+-------+------+



## 6. Construct Hourly Airport Taxi Demand

To align taxi activity with hourly flight schedules, the cleaned airport pickup records are aggregated to the airport-hour level.

Taxi demand is defined as the number of Yellow Taxi pickups originating from each airport during each hour. This produces a directly interpretable demand measure and establishes a common temporal resolution for the subsequent merge with flight activity.

In [19]:
# Create hourly timestamp for each airport pickup
airport_taxi = (
    airport_taxi
    .withColumn(
        "pickup_hour",
        F.date_trunc("hour", F.col("tpep_pickup_datetime"))
    )
)

airport_taxi.select(
    "airport",
    "tpep_pickup_datetime",
    "pickup_hour"
).show(10, truncate=False)

+-------+--------------------+-------------------+
|airport|tpep_pickup_datetime|pickup_hour        |
+-------+--------------------+-------------------+
|JFK    |2025-01-01 00:51:41 |2025-01-01 00:00:00|
|JFK    |2025-01-01 00:55:44 |2025-01-01 00:00:00|
|JFK    |2025-01-01 00:04:29 |2025-01-01 00:00:00|
|LGA    |2025-01-01 00:02:20 |2025-01-01 00:00:00|
|LGA    |2025-01-01 00:08:07 |2025-01-01 00:00:00|
|LGA    |2025-01-01 00:24:51 |2025-01-01 00:00:00|
|LGA    |2025-01-01 00:11:59 |2025-01-01 00:00:00|
|LGA    |2025-01-01 00:39:59 |2025-01-01 00:00:00|
|JFK    |2025-01-01 00:35:44 |2025-01-01 00:00:00|
|JFK    |2025-01-01 00:32:51 |2025-01-01 00:00:00|
+-------+--------------------+-------------------+
only showing top 10 rows


In [20]:
# Aggregate airport pickups to airport-hour demand
hourly_taxi_demand = (
    airport_taxi
    .groupBy("airport", "pickup_hour")
    .agg(
        F.count("*").alias("taxi_pickups")
    )
    .orderBy("airport", "pickup_hour")
)

hourly_taxi_demand.show(20, truncate=False)

+-------+-------------------+------------+
|airport|pickup_hour        |taxi_pickups|
+-------+-------------------+------------+
|JFK    |2025-01-01 00:00:00|233         |
|JFK    |2025-01-01 01:00:00|74          |
|JFK    |2025-01-01 02:00:00|72          |
|JFK    |2025-01-01 03:00:00|20          |
|JFK    |2025-01-01 04:00:00|23          |
|JFK    |2025-01-01 05:00:00|87          |
|JFK    |2025-01-01 06:00:00|168         |
|JFK    |2025-01-01 07:00:00|126         |
|JFK    |2025-01-01 08:00:00|110         |
|JFK    |2025-01-01 09:00:00|105         |
|JFK    |2025-01-01 10:00:00|119         |
|JFK    |2025-01-01 11:00:00|158         |
|JFK    |2025-01-01 12:00:00|182         |
|JFK    |2025-01-01 13:00:00|256         |
|JFK    |2025-01-01 14:00:00|323         |
|JFK    |2025-01-01 15:00:00|352         |
|JFK    |2025-01-01 16:00:00|374         |
|JFK    |2025-01-01 17:00:00|396         |
|JFK    |2025-01-01 18:00:00|344         |
|JFK    |2025-01-01 19:00:00|411         |
+-------+--

In [21]:
# Check the number of observed airport-hour combinations
hourly_count = hourly_taxi_demand.count()

print("Observed airport-hour rows:", hourly_count)
print("Expected airport-hour rows:", 181 * 24 * 2)

hourly_taxi_demand.groupBy("airport").count().show()

Observed airport-hour rows: 8423
Expected airport-hour rows: 8688
+-------+-----+
|airport|count|
+-------+-----+
|    LGA| 4082|
|    JFK| 4341|
+-------+-----+



In [22]:
# Summarise hourly airport taxi demand
hourly_taxi_demand.groupBy("airport").agg(
    F.count("*").alias("n_hours"),
    F.round(F.mean("taxi_pickups"), 2).alias("mean_pickups"),
    F.min("taxi_pickups").alias("min_pickups"),
    F.expr("percentile_approx(taxi_pickups, 0.5)").alias("median_pickups"),
    F.max("taxi_pickups").alias("max_pickups")
).show()

+-------+-------+------------+-----------+--------------+-----------+
|airport|n_hours|mean_pickups|min_pickups|median_pickups|max_pickups|
+-------+-------+------------+-----------+--------------+-----------+
|    LGA|   4082|       155.5|          1|           166|        578|
|    JFK|   4341|      224.78|          1|           203|        741|
+-------+-------+------------+-----------+--------------+-----------+



### 6.1 Complete Airport-Hour Panel

Aggregation only produces rows for hours in which at least one Yellow Taxi pickup was observed. To avoid excluding zero-demand periods, a complete hourly panel is constructed for both JFK and LGA over the study period.

Airport-hour combinations absent from the aggregated trip records are assigned zero observed Yellow Taxi pickups. This ensures that subsequent comparisons with flight activity use a consistent temporal panel.

In [25]:
# Construct a complete calendar-hour grid for the study period.
# We generate date × hour explicitly to avoid daylight-saving-time
# effects when constructing local NYC clock hours.

date_grid = (
    spark.sql("""
        SELECT explode(
            sequence(
                to_date('2025-01-01'),
                to_date('2025-06-30'),
                interval 1 day
            )
        ) AS pickup_date
    """)
)

hour_of_day = spark.range(24).withColumnRenamed("id", "hour_of_day")

hour_grid = (
    date_grid
    .crossJoin(hour_of_day)
    .withColumn(
        "pickup_hour",
        F.to_timestamp_ntz(
            F.concat(
                F.date_format("pickup_date", "yyyy-MM-dd"),
                F.lit(" "),
                F.lpad(F.col("hour_of_day").cast("string"), 2, "0"),
                F.lit(":00:00")
            )
        )
    )
    .select("pickup_hour")
)

print("Hourly rows:", hour_grid.count())
# Create airport dimension
airport_grid = spark.createDataFrame(
    [("JFK",), ("LGA",)],
    ["airport"]
)

# Full airport-hour panel
complete_panel = airport_grid.crossJoin(hour_grid)

print("Complete airport-hour panel rows:", complete_panel.count())

Hourly rows: 4344
Complete airport-hour panel rows: 8688


In [26]:
# Join observed demand onto the complete airport-hour panel
hourly_taxi_complete = (
    complete_panel
    .join(
        hourly_taxi_demand,
        on=["airport", "pickup_hour"],
        how="left"
    )
    .fillna({"taxi_pickups": 0})
    .orderBy("airport", "pickup_hour")
)

print("Final airport-hour rows:", hourly_taxi_complete.count())

hourly_taxi_complete.groupBy("airport").count().show()

Final airport-hour rows: 8688
+-------+-----+
|airport|count|
+-------+-----+
|    JFK| 4344|
|    LGA| 4344|
+-------+-----+



## 7. Validate and Save Final Airport Taxi Demand Panel

We validate the complete airport-hour panel by checking the study-period coverage, the number of zero-demand hours, and the distribution of hourly taxi demand at JFK and LGA.

In [27]:
# Check the final airport-hour time range
hourly_taxi_complete.select(
    F.min("pickup_hour").alias("min_pickup_hour"),
    F.max("pickup_hour").alias("max_pickup_hour")
).show(truncate=False)

+-------------------+-------------------+
|min_pickup_hour    |max_pickup_hour    |
+-------------------+-------------------+
|2025-01-01 00:00:00|2025-06-30 23:00:00|
+-------------------+-------------------+



In [28]:
# Count zero-demand airport-hours
zero_demand_summary = (
    hourly_taxi_complete
    .groupBy("airport")
    .agg(
        F.count("*").alias("total_hours"),
        F.sum(
            F.when(F.col("taxi_pickups") == 0, 1).otherwise(0)
        ).alias("zero_demand_hours")
    )
    .withColumn(
        "zero_demand_pct",
        F.round(
            F.col("zero_demand_hours") / F.col("total_hours") * 100,
            2
        )
    )
    .orderBy("airport")
)

zero_demand_summary.show(truncate=False)

+-------+-----------+-----------------+---------------+
|airport|total_hours|zero_demand_hours|zero_demand_pct|
+-------+-----------+-----------------+---------------+
|JFK    |4344       |3                |0.07           |
|LGA    |4344       |262              |6.03           |
+-------+-----------+-----------------+---------------+



In [29]:
# Summarise hourly taxi demand by airport
demand_summary = (
    hourly_taxi_complete
    .groupBy("airport")
    .agg(
        F.count("*").alias("n_hours"),
        F.round(F.mean("taxi_pickups"), 2).alias("mean_pickups"),
        F.expr("percentile_approx(taxi_pickups, 0.5)").alias("median_pickups"),
        F.round(F.stddev("taxi_pickups"), 2).alias("sd_pickups"),
        F.min("taxi_pickups").alias("min_pickups"),
        F.max("taxi_pickups").alias("max_pickups")
    )
    .orderBy("airport")
)

demand_summary.show(truncate=False)

+-------+-------+------------+--------------+----------+-----------+-----------+
|airport|n_hours|mean_pickups|median_pickups|sd_pickups|min_pickups|max_pickups|
+-------+-------+------------+--------------+----------+-----------+-----------+
|JFK    |4344   |224.62      |203           |140.84    |0          |741        |
|LGA    |4344   |146.12      |155           |110.18    |0          |578        |
+-------+-------+------------+--------------+----------+-----------+-----------+



In [30]:
# Save the final processed airport-hour taxi demand panel
OUTPUT_PATH = "../data/processed/hourly_airport_taxi_demand.parquet"

(
    hourly_taxi_complete
    .write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)

# Reload the saved dataset to verify that it was written correctly
saved_taxi_demand = spark.read.parquet(
    "../data/processed/hourly_airport_taxi_demand.parquet"
)

print("Saved rows:", saved_taxi_demand.count())
print("Saved columns:", len(saved_taxi_demand.columns))

saved_taxi_demand.groupBy("airport").count().orderBy("airport").show()

print("Saved processed taxi demand data to:")
print(OUTPUT_PATH)

Saved rows: 8688
Saved columns: 3
+-------+-----+
|airport|count|
+-------+-----+
|    JFK| 4344|
|    LGA| 4344|
+-------+-----+

Saved processed taxi demand data to:
../data/processed/hourly_airport_taxi_demand.parquet
